<a href="https://colab.research.google.com/github/Giangysam/Correlations/blob/main/Diagnosi_Cause_Difetto_LRLasso_RF_XGB%2BSHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Caricamento librerie
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)
import shap
import matplotlib.pyplot as plt

# CONFIGURAZIONE
CAT_COLS = ['R', 'W', 'V', 'Z', 'AA'] # Explicitly defined categorical columns
COL_ORDINALE = 'R' # Numeric column, OHE destroy it
COLS_DA_ELIMINARE = ['SOGGETTO', 'STAT1'] # Explicitly defined static/numerical columns
CSV_PATH = 'https://raw.githubusercontent.com/Giangysam/Correlations/main/DATI_RIS_coperti.xlsx'
TARGET_COLUMN = 'TARGET'
N_FOLDS = 5
N_REPEATS = 3
RANDOM_STATE = 42
RECALL_TARGET = 0.70
PRECISION_FLOOR_WARNING = 0.10

# Caricamento dei dati
def carica_dati(path=CSV_PATH, target_col=TARGET_COL, verbose=True):
    df = pd.read_excel(path)
    if target_col not in df.columns:
        raise ValueError(f"Colonna target '{target_col}' non trovata. Colonne disponibili: {list(df.columns)}")
    presenti = [c for c in COLS_DA_ELIMINARE if c in df.columns]
    mancanti = [c for c in COLS_DA_ELIMINARE if c not in df.columns]
    if mancanti:
      print(f'ATTENZIONE: colonne da eliminare non trovate nel file (controlla i nomi): {mancanti}')
    df = df.drop(columns=COLS_DA_ELIMINARE, errors='ignore')

    y = df[target_col]
    X = df.drop(columns=[target_col]).copy()

    return X, y

# Categoriche -> stringa (evita TypeError nel OneHotEncoder con ID misti)
    for col in [c for c in CAT_COLS if c in X.columns]:
        X[col] = X[col].astype(str).str.strip()

    # Tutto il resto -> numerico
    for col in [c for c in X.columns if c not in CAT_COLS]:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    # Indici allineati e contigui: serve a .iloc nel k-fold
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    if verbose:
        print(f"Righe: {len(X)}, colonne feature: {X.shape[1]}")
        print("Distribuzione target:")
        print(y.value_counts().rename("conteggio").to_string())
        print(f"Proporzione classe positiva: {y.mean():.3f}")
        print()

    return X, y


def individua_colonne(X, cat_cols=CAT_COLS):
    """Separa le colonne in nominali (cat_cols), binarie e continue.
    La COL_ORDINALE viene trattata come continua (non one-hot)."""
    categoriche = [c for c in cat_cols if c in X.columns]
    numeriche = [c for c in X.columns if c not in categoriche]

    binarie, continue_ = [], []
    for c in numeriche:
        valori = X[c].dropna().unique()
        if len(valori) <= 2 and set(np.unique(valori)).issubset({0, 1}):
            binarie.append(c)
        else:
            continue_.append(c)

    return categoriche, continue_, binarie


# ============================================================
# PIPELINE — ognuna e' una funzione factory: viene ricostruita
# ad ogni fold cosi' scale_pos_weight (XGB) si calcola solo sul
# training fold, senza leakage dal test fold.
# ============================================================
def _preprocessore_con_scaling(categoriche, colonne_continue, colonne_binarie):
    return ColumnTransformer(transformers=[
        ("num", Pipeline([('imputer', SimpleImputer(strategy='median')),
                             ('scaler', StandardScaler())]), colonne_continue),
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoriche),
        ('bin', SimpleImputer(strategy='most_frequent'), colonne_binarie),
        ], remainder='drop')

def _preprocessore_no_scaling(categoriche, colonne_continue, colonne_binarie):
    return ColumnTransformer(transformers=[
        ("num", SimpleImputer(strategy='median'), colonne_continue), # no scaling per RF e XGB, che non ne hanno bisogno
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoriche),
        ('bin', SimpleImputer(strategy='most_frequent'), colonne_binarie),
        ], remainder='drop')


def costruisci_pipeline_lasso(categoriche, colonne_continue, colonne_binarie, y_train=None):
    preprocessor = _preprocessore_con_scaling(categoriche, colonne_continue, colonne_binarie)
    clf = LogisticRegression(
        solver="liblinear", l1_ratio=1.0, C=0.1,
        class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
    return Pipeline([("prep", preprocessor), ("clf", clf)])

def costruisci_pipeline_rf(categoriche, colonne_continue, colonne_binarie, y_train=None):
    preprocessor = _preprocessore_no_scaling(categoriche, colonne_continue, colonne_binarie)
    clf = RandomForestClassifier(
        n_estimators=700, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("prep", preprocessor), ("clf", clf)])

def costruisci_pipeline_xgb(categoriche, colonne_continue, colonne_binarie, y_train=None):
    preprocessor = _preprocessore_no_scaling(categoriche, colonne_continue, colonne_binarie)
    # FIX: senza scale_pos_weight, XGBoost non compensa lo sbilanciamento delle
    # classi come fanno RF e Lasso (class_weight="balanced") -> confronto ingiusto.
    if y_train is not None:
        n_pos = int(np.sum(y_train == 1))
        n_neg = int(np.sum(y_train == 0))
        scale_pos_weight = n_neg / max(n_pos, 1)
    else:
        scale_pos_weight = 1.0
    clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, eval_metric='logloss',
        enable_categorical=False
    )
    return Pipeline([("prep", preprocessor), ("clf", clf)])

MODELLI = {
    "RandomForest": costruisci_pipeline_rf,
    "XGBoost": costruisci_pipeline_xgb,
    "Lasso": costruisci_pipeline_lasso,
}

# ============================================================
# VALUTAZIONE CV + SELEZIONE MODELLO MIGLIORE
# ============================================================
def valuta_con_kfold(factory, X, y, categoriche, colonne_continue, colonne_binarie,
                      n_folds=N_FOLDS, n_repeats=N_REPEATS):
    rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=n_repeats, random_state=RANDOM_STATE)
    aucs, f1s, precisions, recalls = [], [], [], []
    for train_idx, test_idx in rskf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Costruzione del modello per ogni fold
        pipe = factory(categoriche, colonne_continue, colonne_binarie, y_train)
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        if hasattr(pipe, 'predict_proba'):
            y_proba = pipe.predict_proba(X_test)[:, 1]
            aucs.append(roc_auc_score(y_test, y_proba))
        else:
            aucs.append(np.nan)
        f1s.append(f1_score(y_test, y_pred, zero_division=0))
        precisions.append(precision_score(y_test, y_pred, zero_division=0))
        recalls.append(recall_score(y_test, y_pred, zero_division=0))
    return {
        "auc_mean": np.nanmean(aucs), "auc_std": np.nanstd(aucs),
        "f1_mean": np.mean(f1s), "f1_std": np.std(f1s),
        "precision_mean": np.mean(precisions), "precision_std": np.std(precisions),
        "recall_mean": np.mean(recalls), "recall_std": np.std(recalls),
    }


def confronta_modelli(X, y, categoriche, colonne_continue, colonne_binarie,
                      recall_minimo=RECALL_TARGET, precision_minima=PRECISION_FLOOR_WARNING):
    righe = []
    for nome, factory in MODELLI.items():
        print(f"Valutazione {nome} ({N_FOLDS} fold x {N_REPEATS} repeat)...")
        res = valuta_con_kfold(factory, X, y, categoriche, colonne_continue, colonne_binarie)
        res["modello"] = nome
        righe.append(res)

    tabella = pd.DataFrame(righe).set_index("modello")
    tabella = tabella[["recall_mean", "recall_std", "precision_mean", "precision_std",
                        "auc_mean", "auc_std", "f1_mean", "f1_std"]]
    tabella.to_csv('risultati_confronto_modelli.csv')   # Aggiunta

    #Visualizzazione grafica
    plt.figure(figsize=(10,6))
    tabella[['recall_mean', 'precision_mean', 'f1_mean', 'auc_mean']]
    plt.title('Confronto Metriche Modelli')
    plt.ylabel('Valore')
    plt.xticks(rotation=45)
    plt.tight_layout()

    print("\n" + "=" * 70)
    print("CONFRONTO MODELLI")
    print("=" * 70)
    print(tabella.round(3).to_string())

    # Tra i modelli che rispettano SIA il recall minimo SIA una precision minima
        # accettabile, si prende quello con recall piu' alto. Se nessuno rispetta
        # entrambi i vincoli, non si sceglie in automatico un modello che potrebbe
        # limitarsi a prevedere quasi sempre "difetto".

    candidati = tabella[(tabella['recall_mean'] >= recall_minimo) & (tabella['precision_mean'] >= precision_minima)]
    if candidati.empty:
        print(f'\nATTENZIONE: nessun modello raggiunge recall >= {recall_minimo: .0%}'
              f'e precision >= {precision_minima:.0%} contemporaneamente.')
        migliore = tabella.sort_values('recall_mean', ascending=False).index[0]
        print(f'Recall più alto in assoluto (sotto la soglia di precision): {migliore}'
              f' -> controllare la confusion matrix prima di usarlo')
    else:
        migliore = candidati.sort_values('recall_mean', ascending=False).index[0]
        print(f'\nMODELLO CONSIGLIATO: {migliore}'
             f'(rispetta recall >= {recall_minimo:.0%} e precision >= {precision_minima:.0%})')

    riga_migliore = tabella.loc[migliore]
    print(f"\n>>> MODELLO CONSIGLIATO: {migliore} "
          f"(recall={riga_migliore['recall_mean']:.3f}, precision={riga_migliore['precision_mean']:.3f})"
          f'f1={riga_migliore['f1_mean']:.3f} auc={riga_migliore['auc_mean']:.3f}')

    return tabella, migliore

def visualizza_metriche_confronto(tabella):
    fig, axes = plt.subplots(2, 2, figsize=(12,10))

    # Recall
    axes[0,0].bar(tabella.index, tabella['recall_mean'], yerr=tabella['recall_std'])
    axes[0,0].set_title('Recall Media per Modello')
    axes[0,0].set_ylabel('Recall')
    axes[0,0].tick_params(axis='x', rotation=45)

    # Precision
    axes[0,1].bar(tabella.index, tabella['precision_mean'], yerr=tabella['precision_std'])
    axes[0,1].set_title('Precisione Media per Modello')
    axes[0,1].set_ylabel('Precisione')
    axes[0,1].tick_params(axis='x', rotation=45)

    # AUC
    axes[1,0].bar(tabella.index, tabella['auc_mean'], yerr=tabella['auc_std'])
    axes[1,0].set_title('AUC Media per Modello')
    axes[1,0].set_ylabel('AUC')
    axes[1,0].tick_params(axis='x', rotation=45)

    # F1-Score
    axes[1,1].bar(tabella.index, tabella['f1_mean'], yerr=tabella['f1_std'])
    axes[1,1].set_title('F1-Score Medio per Modello')
    axes[1,1].set_ylabel('F1-Score')
    axes[1,1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig("metriche_confronto_modelli.png")
    plt.show()

def calcola_matrice_confusione(model, X_test, y_test, nome_modello):
    y_pred = model.predict(X_test)

    # Calcolo della cm
    cm = confusion_matrix(y_test, y_pred)
    print('Confusion Matrix:')
    print()
    print(cm)

    # Visualizzazione della cm
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo', 'Positivo'])
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(f'Matrice di confusione - {nome_modello}')
    plt.tight_layout()
    plt.savefig(f'matrice_confusione_{nome_modello}.png')
    plt.show()

    return cm

# ============================================================
# FEATURE IMPORTANCE — SHAP per modelli tree-based (RF, XGB),
# coefficienti per Lasso. Stessa funzione SHAP riusata per RF/XGB.
# NOTA METODOLOGICA: SHAP spiega cosa il modello associa al difetto nei dati
# osservati, non un rapporto causale. Prima di tradurre le feature in cima alla
# classifica in "configurazioni di lavorazione da adottare", verificare che non
# ci sia confondimento (es. una base/postazione correlata col lotto).
# ============================================================
def analisi_shap(pipe, X, y, nome_modello):
    pipe.fit(X, y)
    modello = pipe.named_steps["clf"]
    X_transformed = pipe.named_steps["prep"].transform(X)
    feature_names = np.array(pipe.named_steps["prep"].get_feature_names_out())

    explainer = shap.TreeExplainer(modello)
    shap_values = explainer.shap_values(X_transformed)

    sv = None
    if isinstance(shap_values, list):
              sv = shap_values[1]
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            if arr.shape[2] == 2:
                sv = arr[:, :, 1]
            elif arr.shape[1] == 2:
                sv = arr[:, 1, :]
            else:
                raise ValueError (f'Shap non riconosciuta: {arr.shape}')
        elif arr.ndim == 2:
            sv = arr
        else:
            raise ValueError(f'Shap non riconosciuta: {arr.shape}')


    if sv.shape[1] != len(feature_names):
        raise ValueError(f"Discrepanza tra numero di feature SHAP ({sv.shape[1]}) e nomi feature ({len(feature_names)})")

    plt.figure()
    shap.summary_plot(sv, X_transformed, feature_names=feature_names, show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(f"shap_summary_{nome_modello}.png", dpi=150, bbox_inches="tight")
    plt.close()

    importanza_media = np.abs(sv).mean(axis=0)
    tabella = pd.DataFrame({"feature": feature_names, "importanza_shap_media": importanza_media})
    tabella = tabella.sort_values("importanza_shap_media", ascending=False)
    tabella.to_csv(f"shap_importanza_{nome_modello}.csv", index=False)
    return tabella


def analisi_lasso_coefficienti(pipe, X, y):
    pipe.fit(X, y)
    modello = pipe.named_steps["clf"]
    feature_names = pipe.named_steps["prep"].get_feature_names_out()
    coef = modello.coef_[0]
    tabella = pd.DataFrame({"feature": feature_names, "coefficiente": coef})
    tabella["abs_coef"] = tabella["coefficiente"].abs()
    tabella = tabella[tabella["abs_coef"] > 1e-6].sort_values("abs_coef", ascending=False)
    tabella.to_csv("lasso_coefficienti.csv", index=False)
    return tabella


# ============================================================
# CORRELAZIONE UNIVARIATA feature -> target
# (indipendente dal modello: cosa e' davvero associato al difetto)
# ============================================================
def cramers_v(x, y):
    tabella_contingenza = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabella_contingenza)[0]
    n = tabella_contingenza.sum().sum()
    r, k = tabella_contingenza.shape
    denom = min(r - 1, k - 1)
    if denom == 0 or n == 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)


def analisi_correlazione_target(X, y, categoriche, colonne_continue, colonne_binarie, col_ordinale=COL_ORDINALE):
    righe = []
    for col in colonne_continue:
        valori = pd.to_numeric(X[col], errors='coerce')
        mask = valori.notna()
        if mask.sum() < 3 or valori[mask].nunique() < 2:
            continue
        if col == col_ordinale:
            r, p = spearman(y[mask], valori[mask])
            righe.append({'feature': col, 'tipo': 'ordinale', 'associazione': abs(r),
                          'metrica': 'Spearman rho', 'p_value': p})
        else:
            r, p = pointbiserialr(y[mask], valori[mask])
            righe.append({"feature": col, "tipo": "continua", "associazione": abs(r), "metrica": "point-biserial r", "p_value": p})
    for col in list(colonne_binarie) + list(categoriche):
        v = cramers_v(X[col].astype(str), y)
        if not np.isnan(v):
            tipo = 'categorica' if col in categoriche else 'binaria'
            righe.append({"feature": col, "tipo": "categorica", "associazione": v, "metrica": "Cramer's V", "p_value": np.nan})

    tabella = pd.DataFrame(righe).sort_values("associazione", ascending=False)
    tabella.to_csv("correlazione_feature_target.csv", index=False)
    return tabella


# ============================================================
# CONFRONTO INCROCIATO DELLE CLASSIFICHE DI IMPORTANZA
# ============================================================
def confronta_importanze(tab_rf, tab_xgb, tab_lasso, tab_corr, top_n=15):
    top_rf = set(tab_rf.head(top_n)["feature"])
    top_xgb = set(tab_xgb.head(top_n)["feature"])
    top_lasso = set(tab_lasso.head(top_n)["feature"])
    top_corr = set(tab_corr.head(top_n)["feature"])

    print(f"\n--- Feature nella top {top_n} di TUTTI e tre i modelli ---")
    comuni_modelli = top_rf & top_xgb & top_lasso
    print(sorted(comuni_modelli) if comuni_modelli else "  Nessuna sovrapposizione diretta.")

    print(f"\n--- Feature nella top {top_n} dei modelli MA NON tra le piu' correlate individualmente ---")
    print("  (segnale possibile di interazioni tra variabili, non di effetto diretto)")
    solo_modelli = comuni_modelli - top_corr
    print(sorted(solo_modelli) if solo_modelli else "  Nessuna.")

    print(f"\n--- Feature tra le piu' correlate MA ignorate da tutti i modelli ---")
    print("  (possibile segnale che i modelli non la stanno sfruttando bene)")
    solo_corr = top_corr - (top_rf | top_xgb | top_lasso)
    print(sorted(solo_corr) if solo_corr else "  Nessuna.")

    return comuni_modelli


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    X, y = carica_dati(CSV_PATH, TARGET_COL)
    categoriche, colonne_continue, colonne_binarie = individua_colonne(X)

    # Split 80/20 per training e test set (per confronto CV, SHAP e confusion matrix finale)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

    # 1) Confronto e selezione del modello migliore per recall
    tabella_confronto, modello_migliore = confronta_modelli(X_train, y_train, categoriche, colonne_continue, colonne_binarie)

    # 2) Visualizzazione grafica delle metriche
    visualizza_metriche_confronto(tabella_confronto)

    # 3) Fit del modello scelto su X_train, valutazione su X_test e calcolo della confusion matrix
    pipe_finale = MODELLI[modello_migliore](categoriche, colonne_continue, colonne_binarie, y_train)
    pipe_finale.fit(X_train, y_train)
    calcola_matrice_confusione(pipe_finale, X_test, y_test, modello_migliore)

    # 4) Analisi feature importance e correlazione univariata, su TUTTO X, y (non solo training)
    print('\nCalcolo SHAP Random Forest...')
    tab_rf = analisi_shap(costruisci_pipeline_rf(categoriche, colonne_continue, colonne_binarie, y), X, y, "RandomForest")
    print('\nCalcolo SHAP XGBoost...')
    tab_xgb = analisi_shap(costruisci_pipeline_xgb(categoriche, colonne_continue, colonne_binarie, y), X, y, "XGBoost")
    print('\nCalcolo coefficienti Lasso...')
    tab_lasso = analisi_lasso_coefficienti(costruisci_pipeline_lasso(categoriche, colonne_continue, colonne_binarie, y), X, y)

    # 5) Correlazione univariata feature -> target
    print("\nCalcolo correlazione feature-target...")
    tab_corr = analisi_correlazione_target(X, y, categoriche, colonne_continue, colonne_binarie)
    print("\nTop 15 feature per correlazione con il difetto:")
    print(tab_corr.head(15).to_string(index=False))

    # 5) Confronto incrociato
    confronta_importanze(tab_rf, tab_xgb, tab_lasso, tab_corr)

    print(f"\nFile salvati: shap_importanza_RandomForest.csv, shap_importanza_XGBoost.csv, "
          f"lasso_coefficienti.csv, correlazione_feature_target.csv, "
          f"shap_summary_RandomForest.png, shap_summary_XGBoost.png")


1. CARICAMENTO DATI
Righe: 221, Colonne feature: 68
Distribuzione target: {0: 140, 1: 81}
Colonne continue (verranno scalate): 62
Colonne binarie/one-hot (non scalate): 6

2. VALUTAZIONE PERFORMANCE - RANDOM FOREST (K-FOLD)
Fold 1: AUC=0.792  F1=0.625  Precision=0.714  Recall=0.556
Fold 2: AUC=0.910  F1=0.842  Precision=0.800  Recall=0.889
Fold 3: AUC=0.799  F1=0.636  Precision=0.538  Recall=0.778
Fold 4: AUC=0.896  F1=0.778  Precision=0.778  Recall=0.778
Fold 5: AUC=0.833  F1=0.636  Precision=0.538  Recall=0.778
Fold 6: AUC=0.763  F1=0.636  Precision=0.538  Recall=0.778
Fold 7: AUC=0.637  F1=0.375  Precision=0.429  Recall=0.333
Fold 8: AUC=0.793  F1=0.625  Precision=0.714  Recall=0.556
Fold 9: AUC=0.748  F1=0.625  Precision=0.714  Recall=0.556

--- Medie su tutti i fold ---
AUC:       0.797 (+/- 0.077)
F1:        0.642 (+/- 0.121)
Precision: 0.641 (+/- 0.123)
Recall:    0.667 (+/- 0.166)

3. VALUTAZIONE PERFORMANCE - LASSO (K-FOLD)
Fold 1: AUC=0.764  F1=0.609  Precision=0.500  Recall=

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 3: AUC=0.806  F1=0.640  Precision=0.500  Recall=0.889
Fold 4: AUC=0.861  F1=0.762  Precision=0.667  Recall=0.889
Fold 5: AUC=0.701  F1=0.667  Precision=0.500  Recall=1.000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 6: AUC=0.733  F1=0.583  Precision=0.467  Recall=0.778
Fold 7: AUC=0.704  F1=0.609  Precision=0.500  Recall=0.778
Fold 8: AUC=0.926  F1=0.824  Precision=0.875  Recall=0.778


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 9: AUC=0.733  F1=0.636  Precision=0.538  Recall=0.778

--- Medie su tutti i fold ---
AUC:       0.795 (+/- 0.084)
F1:        0.669 (+/- 0.074)
Precision: 0.564 (+/- 0.122)
Recall:    0.852 (+/- 0.091)

4. ANALISI DIAGNOSTICA - SHAP (Random Forest su tutti i dati)


100%|===================| 440/442 [00:17<00:00]       

Salvato: shap_summary.png (le 20 variabili più importanti, con direzione dell'effetto)
Salvato: shap_importanza.csv (ranking completo)

Top 15 variabili per importanza SHAP:
         feature  importanza_shap_media
         num__AR               0.022983
          num__C               0.020345
         num__AS               0.018747
         num__AH               0.018333
num__TIRO_G4_USC               0.015977
         num__AG               0.014256
          num__B               0.012995
         num__AU               0.010674
         num__AQ               0.009827
         num__BN               0.008144
         num__AX               0.007822
          num__I               0.007042
         num__AI               0.006809
         num__AD               0.006772
         num__BA               0.006688

5. ANALISI DIAGNOSTICA - LASSO (coefficienti su tutti i dati)

Lasso: 11/220 feature con coefficiente non-zero
Salvato: lasso_coefficienti.csv

Top 15 coefficienti (segno positivo = aum

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
